<h3>Index</h3>
<ul>
  <li><a href="#analyse-exploratoire">1. Analyse exploratoire</a></li>
  <li><a href="#hypothese">2. Hypothèse</a></li>
  <li><a href="#nettoyage">3. Nettoyage et fusion des fichiers</a></li>
  <li><a href="#creer-features">4. Créer les features</a></li>
  <li><a href="#selectionner-features">5. Selectionner les Features</a></li>
  <li><a href="#explication-features">6. Explication des Features</a></li>
  <li><a href="#train-test">7. Train/Test</a></li>
  <li><a href="#preprocesseur">8. Préprocesseur</a></li>
  <li><a href="#explication-stratification">9. Explication de la statification train/test et du préprocésseur</a></li>
  <li><a href="#modelisation">10. Modelisation</a></li>
  <li><a href="#analyse-modeles">11. Analyse des modèles</a></li>
  <li><a href="#recherche-hyper-parametres">12. Recherche d'hyper parametres</a></li>
  <li><a href="#analyse-shap">13. Analyse Shap</a></li>
</ul>

### Imports

In [1]:
import pandas as pd
from IPython.display import display,clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import scipy.stats as stats
import numpy as np
import ipywidgets as widgets
import statsmodels.api as sm
import sys, os
import ipywidgets as widgets


#Import des modules 
sys.path.append(os.path.abspath("src/Modules"))
import exploration.exploration as exploration
import exploration.clean_fusion as clean_fusion
import features.new_features as new_features
import features.features_selection as features_selection
import models__analyses.train_test as train_test


import models__analyses.model as model
import models__analyses.etude_model as etude_model

import optimisations__analyses.shap_analyse as shap_analyse
import optimisations__analyses.hyperparameters_interface as hyperparameters_interface

#Import librairies pour les modules
import inspect
from matplotlib.lines import Line2D
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix
from IPython.display import display, clear_output


In [2]:
df_eval = pd.read_csv("datas/extrait_eval.csv")
df_sirh = pd.read_csv("datas/extrait_sirh.csv")
df_sondage = pd.read_csv("datas/extrait_sondage.csv")

# 1. Analyse exploratoire
<div id="analyse-exploratoire"></div>

In [3]:
from exploration.exploration import explorer, analyser_attrition_globale, bouton_explorer
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Bouton Exploration Globale
bouton = bouton_explorer()
sortie = widgets.Output()

def au_clic(_):
    with sortie:
        clear_output(wait=True)
        explorer(df_sirh, df_sondage, df_eval)

bouton.on_click(au_clic)

display(widgets.VBox([widgets.Label("→ 1. Exploration Standard"), bouton, sortie]))


# 2. Bouton Analyse Globale de l'Attrition

#Création du bouton
bouton_attrition_global = widgets.Button(
    description="Analyser l'Attrition Globale",
    button_style="success",
    icon="globe",
    layout=widgets.Layout(width="280px", height="40px"),
)
sortie_attrition_global = widgets.Output()
#clear outpout pour relancer
def au_clic_attrition_global(_):
    with sortie_attrition_global:
        clear_output(wait=True)
        analyser_attrition_globale(df_sirh, df_sondage, df_eval)

bouton_attrition_global.on_click(au_clic_attrition_global)
display(widgets.VBox([widgets.Label("→ 2. Analyse Avancée de l'Attrition"), bouton_attrition_global, sortie_attrition_global]))

# 2. Hypothèse
<div id="hypothese"></div>



<div style="font-size:18px;line-height:1.55;color:#1c2333;background:#fffdf9;border:1px solid #e6dfd4;border-radius:18px;padding:28px 32px;font-family:Georgia,'Iowan Old Style',Palatino,serif;">

<p style="margin:0 0 8px;font-family:'Segoe UI',system-ui,sans-serif;font-size:12px;letter-spacing:.16em;text-transform:uppercase;color:#c45c2a;font-weight:700;">
Hypothèse · Attrition
</p>

<h3 style="margin:0 0 16px;font-size:26px;line-height:1.25;font-weight:600;">
Un profil junior peu ancré, mais ce n'est pas une cause unique.
</h3>

<p style="margin:0 0 14px;">
L’attrition ne se répartit pas au hasard : elle se concentre chez les salariés
<strong>en début de parcours</strong> → plus jeunes, niveau de poste bas, salaire plus faible,
peu d’ancienneté dans l’entreprise, dans le poste et sous le responsable actuel.
</p>

<p style="margin:0 0 14px;">
Les matrices Pearson et Spearman le montrent directement : âge, revenu, expérience totale,
ancienneté, années dans le poste / sous le responsable et niveau hiérarchique forment un
<strong>bloc rouge</strong>. Elles varient ensemble. On ne peut donc pas dire
« c’est le salaire » ou « c’est l’âge » : la matrice dit que c’est <em>le même axe</em>.
</p>

<p style="margin:0 0 14px;">
La ligne <strong>a_quitte_l_entreprise</strong> de ces mêmes matrices reste claire :
le départ est associé à ce bloc, les vraiables allant de environ -17 à envrion −0,20</span>.<br>
Les satisfactions sont à l’écart de ce bloc (cases pâles) : et nous pourront croiser ces signaux avec des des indicateurs fortement corrélés à la cible, <br>
notament car les causes de l'attrition sont multicausales et que dans le cadre d'une recherche de causes qualitatives, ces indicateurs unifiés produirait un biais et empecherait la variance.<br>

Les variables qualitatives telques<strong>+ 0,24</strong> les heures supplémentaires et le poste également a envrion + 0,24 ou le sstatus marital a +0,17 sont des atouts <em>qualitatifs</em> 
 par opposition à l'age, la rémunération, le nombre d'année dans le poste, participation pee ou nombre d'année dans l'entreprise.
</p>

<p style="margin:0 0 14px;">
Le diplôme, le genre et la date de dernière promotion on une influence quasiment null.
</p>

<p style="margin:0;padding-top:14px;border-top:1px solid #e6dfd4;color:black;font-size:16px;">
Hypothèse : un <strong style="color:#1c2333;">profil junior peu ancré</strong>, lu dans les
<strong style="color:#1c2333;">bloc de corrélations, les courbe KDE et les dérivées des indicateurs vers y</strong> → sont plusieurs facteurs liés entre eux,
mais pas une cause unique.
</p>

</div>

# 3. Nettoyage et fusion des fichiers
<div id="nettoyage"></div>

In [4]:
def get_sources():
    return df_sirh, df_sondage, df_eval

def save_clean(res):
    global df_clean
    df_clean = res

# Affichage du bouton interactif
clean_fusion.bouton_nettoyer_fusionner(get_sources, save_clean)

# 4. Créer les features
<div id="creer-features"></div>

In [5]:
def get_df():
    return df_clean

def save_df(res):
    global df_clean, X, y
    df_clean = res
    X, y = new_features.preparer_X_y(df_clean)
    assert X is not None and y is not None, "❌ Erreur : X ou y n'a pas pu être créé."
    assert len(X) == len(y), f"❌ Erreur de dimension : X a {len(X)} lignes mais y a {len(y)} lignes."
    assert X.shape[0] > 0, "❌ Erreur : Le DataFrame X est vide."
    X.to_csv("X.csv", index=False)
    y.to_csv("y.csv", index=False)
    print(f"✅ Validation réussie : X ({X.shape[1]} features) et y sont prêts, synchronisés et sauvegardés !")

display(new_features.bouton_creer_features(get_df, save_df))

In [6]:
def get_current_df():
    return df_clean

def set_current_df(new_df):
    global df_clean
    df_clean = new_df

exploration.bouton_analyse_features(get_current_df, set_current_df)

# 5. Selectionner les Features 
<div id="selectionner-features"></div>

In [7]:

def get_df():
    return df_clean

def au_demarrage(variables_choisies):
    global variables_selectionnees
    variables_selectionnees = variables_choisies
    print("Prêt pour la suite avec :", variables_choisies)

def au_panneau_pret(vb):
    global var_buttons
    var_buttons = vb

display(features_selection.bouton_features_selection(
    get_df,
    au_demarrage,
    target_col="a_quitte_l_entreprise",
    on_panel_ready_callback=au_panneau_pret,
))

# 6. Explication des Features
<div id="explication-features"></div>

<div style="font-size:20px;line-height:1.65;color:#1c2333;background:#fffdf9;border:1px solid #e6dfd4;border-radius:16px;padding:32px 36px;font-family:Georgia,Palatino,serif;">
<p style="margin:0 0 8px;font-family:'Segoe UI',sans-serif;font-size:14px;letter-spacing:.14em;text-transform:uppercase;color:#c45c2a;font-weight:700;">
Quelles familles de features, quelles features on garde → et pourquoi ?
</p>
<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">1. Quatre familles, quatre traitements</p>
<p style="margin:0 0 16px;">
Le modèle ne lit pas toutes les colonnes de la même façon. D’où les 4 groupes :<br>
<strong>Linéaires</strong> → un nombre qui pousse le risque à peu près droit (standardisation).<br>
<strong>Non-linéaires</strong> → une courbe, même légère (standardisation).<br>
<strong>Catégorielles</strong> → du texte (poste, marital…) : une case par valeur (one-hot).<br>
<strong>Booléennes</strong> → oui / non, déjà 0/1 : on les laisse telles quelles.
</p>
<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">2. On retire une variable dans trois cas</p>
<p style="margin:0 0 16px;">
<strong>1. Elle ne parle presque pas du départ.</strong><br>
Genre, diplôme / domaine d’étude, nombre de formations, <code>formations_par_an</code> : ceux qui partent et ceux qui restent se ressemblent. La permutation importance tombe vers 0. Garder ces colonnes n’ajoute que du bruit.
</p>
<p style="margin:0 0 16px;">
<strong>2. Elle redit la même chose qu’une autre déjà là (redondance).</strong><br>
Âge, revenu, expérience totale, ancienneté entreprise / poste / manager et niveau de poste forment un <em>bloc rouge</em> : elles montent ensemble. Ce n’est pas « six causes », c’est <strong>un seul axe</strong> (début de parcours). On en garde une ou deux (ex. années sous le responsable, ou ancienneté entreprise), on retire les jumelles. Sinon le modèle répète le même signal et les coefficients deviennent instables.
</p>
<p style="margin:0 0 16px;">
<strong>3. Elle est un double d’une synthèse déjà créée.</strong><br>
Pas les 4 notes de satisfaction + la globale + le min : une seule suffit (souvent <code>satisfaction_min</code>). Pas <code>poste</code> et <code>niveau</code> et <code>poste_x_niveau</code> en même temps. La feature croisée remplace les deux sources.
</p>
<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">3. La règle en une ligne</p>
<p style="margin:0 0 16px;">
On <strong>garde</strong> si la variable apporte encore une info sur le départ <em>une fois les autres connues</em>.<br>
On <strong>retire</strong> si elle est muette, ou si elle copie un signal déjà pris (âge + expérience, quatre satisfactions, poste + niveau alors que <code>poste_x_niveau</code> existe).
</p>
<p style="margin:0;padding-top:16px;border-top:1px solid #e6dfd4;color:#1c2333;font-size:18px;">
Quatre familles pour le préprocesseur.
Récit fil rouge : junior peu ancré + heures sup / poste + un peu de satisfaction. 
</p> <br>
    <p><b>On enlève ce qui est muet ou en double pour que le modèle apprenne le signal junior, pas le bruit du train.</b></p>
</div>


# 7. Train/Test
<div id="train-test"></div>

In [8]:
from sklearn.model_selection import StratifiedKFold

def get_df():
    return df_clean

def get_var_buttons():
    if 'var_buttons' in globals():
        return var_buttons
    print("⚠️ Le panneau de sélection des features n'a pas été lancé !")
    return None

def on_split_termine(resultats):
    global X_train, X_test, y_train, y_test, features_selectionnees, cv
    X_train = resultats["X_train"]
    X_test = resultats["X_test"]
    y_train = resultats["y_train"]
    y_test = resultats["y_test"]
    features_selectionnees = resultats["features_selectionnees"]
    
    # Récupération dynamique de la graine choisie dans l'interface de split
    seed_dynamique = resultats.get("random_state", 42)
    
    # Création du CV global dynamique pour l'interface d'hyperparamètres
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed_dynamique)

display(train_test.bouton_train_test(get_df, get_var_buttons, on_split_termine, target_col="a_quitte_l_entreprise"))

# 8. Préprocesseur 
<div id="preprocesseur"></div>

In [9]:
# ============================================================
# ÉTAPE 2 — PRÉPROCESSEUR (fit sur le train seulement)
# ============================================================
btn_prep = widgets.Button(
    description="Lancer le préprocesseur",
    button_style="warning",
    layout=widgets.Layout(width="300px", height="40px"),
)
out_prep = widgets.Output()
display(widgets.VBox([btn_prep, out_prep]))

def on_prep(b):
    global preprocessor, X_train_prep, X_test_prep
    with out_prep:
        clear_output()
        if "X_train" not in globals():
            print("Lance d’abord la cellule 1 (train / test).")
            return
        if "var_buttons" not in globals():
            print("Le panneau de sélection des variables n'a pas été initialisé.")
            return
            
        # Récupération de la sélection active par groupe depuis le panneau interactif
        from features.features_selection import obtenir_variables_selectionnees
        
        selection_par_groupe = obtenir_variables_selectionnees(var_buttons, par_groupe=True)
        
        preprocessor = construire_preprocessor(
            colonnes_lineaires=selection_par_groupe.get("lineaires", []),
            colonnes_non_lineaires=selection_par_groupe.get("non_lineaires", []),
            cols_qualitatives=selection_par_groupe.get("qualitatives", []),
            cols_booleennes=selection_par_groupe.get("booleennes", []),
            cols_ratios=selection_par_groupe.get("nouvelles_features", []),
            inclure_lineaires=True,
            inclure_non_lineaires=True,
            inclure_qualitatives=True,
            inclure_booleennes=True,
            inclure_ratios=True,
            X=X_train,
        )
        
        preprocessor.fit(X_train)
        X_train_prep = preprocessor.transform(X_train)
        X_test_prep = preprocessor.transform(X_test)
        
        print("Fit uniquement sur X_train (pas de fuite du test).")
        print(f"X_train transformé : {getattr(X_train_prep, 'shape', type(X_train_prep))}")
        print(f"X_test transformé  : {getattr(X_test_prep, 'shape', type(X_test_prep))}")
        print("\nÉtape 2 OK → passe à la cellule modèles.")

btn_prep.on_click(on_prep)

# 9. Explication de la statification train/test et du préprocésseur
<div id="explication-stratification"></div>

<div style="font-size:20px;line-height:1.65;color:#1c2333;background:#fffdf9;border:1px solid #e6dfd4;border-radius:16px;padding:32px 36px;font-family:Georgia,Palatino,serif;">

<p style="margin:0 0 8px;font-family:'Segoe UI',sans-serif;font-size:14px;letter-spacing:.14em;text-transform:uppercase;color:#c45c2a;font-weight:700;">
Pipeline &amp; préparation des données
</p>

<h3 style="margin:0 0 20px;font-size:30px;line-height:1.25;font-weight:600;">
Préprocesseur, pipeline et cible asymétrique
</h3>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">1. Ordre des étapes</p>

<p style="margin:0 0 16px;background:#f4efe8;padding:14px 16px;border-radius:10px;">
<strong>1.</strong> Sélection des variables (panneau ON/OFF)<br>
<strong>2.</strong> Split train / test <strong>stratifié</strong> (80 / 20)<br>
<strong>3.</strong> Construction du préprocesseur<br>
<strong>4.</strong> Modèles évalués <strong>dans un <code>Pipeline</code> sklearn</strong>
</p>

<p style="margin:0 0 16px;">
Le préprocesseur n'est jamais fit sur le test.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">2.Exemple de Split observé</p>

<p style="margin:0 0 16px;">
<strong>Train :</strong> 1 176 lignes (16,2 % de départs)<br>
<strong>Test :</strong> 294 lignes (16,0 % de départs)<br>
</p>

<p style="margin:0 0 16px;">
La cible <code>a_quitte_l_entreprise</code> est <strong>asymétrique</strong> (~16 % de départs).
On préapre ainsi le train et test sous la forme de <strong>stratifications.</strong>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">3. Préprocesseur</p>

<table style="width:100%;border-collapse:collapse;margin:0 0 16px;font-size:18px;">
<thead>
<tr style="background:#f4efe8;">
<th style="text-align:left;padding:10px 12px;border:1px solid #e6dfd4;">Groupe</th>
<th style="text-align:left;padding:10px 12px;border:1px solid #e6dfd4;">Traitement</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding:10px 12px;border:1px solid #e6dfd4;">Numériques linéaires / non linéaires / ratios</td>
<td style="padding:10px 12px;border:1px solid #e6dfd4;"><code>StandardScaler</code></td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e6dfd4;">Qualitatives</td>
<td style="padding:10px 12px;border:1px solid #e6dfd4;"><code>OneHotEncoder(handle_unknown="ignore")</code></td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e6dfd4;">Booléennes (0/1)</td>
<td style="padding:10px 12px;border:1px solid #e6dfd4;"><code>passthrough</code></td>
</tr>
</tbody>
</table>

<p style="margin:0 0 12px;">
La cellule « Lancer le préprocesseur » sert de <strong>contrôle</strong>.
L'évaluation réelle se fait dans <code>evaluer_modeles</code> :
</p>

<pre style="margin:0;background:#f4efe8;padding:14px 16px;border-radius:10px;font-family:Consolas,Menlo,monospace;font-size:17px;overflow-x:auto;">Pipeline(
  preprocessor  →  classifieur
)</pre>

</div>

# 10. Modelisation
<div id="modelisation"></div>

 ### → Choisir le nombre de modèles à entraîner puis cliquer sur 🚀 **Lancer l'entraînement**

In [10]:
from models__analyses.model import construire_preprocessor
#Ignorer les warnings de futures dépréciations de focntions scikit-learn 
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

display(model.interface_modelisation_etape())

# 11. Analyse des modèles
<div id="analyse-modeles"></div>

### → **1**-Actualiser la liste des modèles entraînés  → **2**-Choisir le nombre top N de modèles à analyser →**3**-Cliquer sur 🚀 **Charger / Actualiser les analyses**

In [11]:
sys.path.append(os.path.abspath('src/Modules'))

display(etude_model.interface_etude_modeles_etape())

<div style="font-size:19px;line-height:1.65;color:#1c2333;background:#fffdf9;border:1px solid #e6dfd4;border-radius:16px;padding:32px 36px;font-family:Georgia,Palatino,serif;">

<p style="margin:0 0 8px;font-family:'Segoe UI',sans-serif;font-size:13px;letter-spacing:.14em;text-transform:uppercase;color:#c45c2a;font-weight:700;">
Diagnostic du modèle
</p>
<p style="margin:0 0 20px;font-family:'Segoe UI',sans-serif;font-size:22px;font-weight:700;color:#1c2333;">
Comment on juge, comment on lit, comment on revient aux features
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">1. Les cinq pastilles → comment elles sont calculées</p>
<p style="margin:0 0 12px;">
Tout part du <strong>ROC-AUC</strong>. On compare le score moyen des plis d’entraînement
(<code>ROC_AUC_train_cv_mean</code>), celui de la validation (<code>ROC_AUC_cv</code>)
et l’écart-type entre plis (<code>ROC_AUC_cv_std</code>).
</p>
<p style="margin:0 0 8px;"><code>Écart_Train_CV = AUC_train_des_plis → AUC_cv</code></p>
<table style="width:100%;border-collapse:collapse;margin:0 0 14px;font-size:16px;">
  <tr style="background:#f4eee6;">
    <th style="text-align:left;padding:8px 10px;">Pastille</th>
    <th style="text-align:left;padding:8px 10px;">Règle</th>
    <th style="text-align:left;padding:8px 10px;">En français</th>
  </tr>
  <tr>
    <td style="padding:8px 10px;background:#d4edda;color:#155724;font-weight:600;">Bien équilibré</td>
    <td style="padding:8px 10px;">aucun des cas ci-dessous</td>
    <td style="padding:8px 10px;">Train et CV proches (±0,05), score pas ridicule</td>
  </tr>
  <tr>
    <td style="padding:8px 10px;background:#f8d7da;color:#721c24;font-weight:600;">Surapprentissage</td>
    <td style="padding:8px 10px;">Écart_Train_CV &gt; 0,05</td>
    <td style="padding:8px 10px;">il récite le train, la CV décroche</td>
  </tr>
  <tr>
    <td style="padding:8px 10px;background:#ffe8cc;color:#7a3e00;font-weight:600;">Sous-apprentissage</td>
    <td style="padding:8px 10px;">Train_CV <strong>et</strong> CV &lt; 0,65</td>
    <td style="padding:8px 10px;">trop simple, même sur le train</td>
  </tr>
  <tr>
    <td style="padding:8px 10px;background:#e8d5f5;color:#4a1c6b;font-weight:600;">Atypique</td>
    <td style="padding:8px 10px;">Écart_Train_CV &lt; −0,05</td>
    <td style="padding:8px 10px;">la CV bat le train : hasard de plis ou fuite</td>
  </tr>
  <tr>
    <td style="padding:8px 10px;background:#fff3cd;color:#856404;font-weight:600;">Instable</td>
    <td style="padding:8px 10px;">écart-type des folds &gt; 0,10</td>
    <td style="padding:8px 10px;">le score saute d’un pli à l’autre</td>
  </tr>
</table>
<p style="margin:0 0 16px;">
Le <strong>test</strong> n’entre pas dans la pastille : c’est le tiroir fermé.
Si la CV est bien et le test non, état alors  bien noté des bouts du train.
Un Recall train excellent + pastille rouge = il a appris le cahier par cœur : / ! \ a ne pas déployer en production.
On lit Recall / Precision / F1 sur le <em>test</em>.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">2. Histogrammes de probabilités (train vs test)</p>
<p style="margin:0 0 16px;">
Hauteur = effectif. Axe = score de départ. Trait rouge = seuil (souvent 0,50).
<span style="color:#4682b4;">Bleu VN</span>,
<span style="color:#daa520;">orange FP</span>,
<span style="color:#cd5c5c;">rouge FN</span>,
<span style="color:#228b22;">vert VP</span>.
Train et test doivent se ressembler. Beaucoup d’orange à droite du seuil = filet large.
Beaucoup de rouge à gauche = départs trop bas → baisser le seuil, au prix de plus d’orange.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">3. Grille « proba vs réel »</p>
<p style="margin:0 0 16px;">
Colonne 0 = restés, colonne 1 = partis. Hauteur = score. Crème = alerte, bleu = pas d’alerte.
Vert = le seuil a raison, rouge = il se trompe.
Si les deux colonnes se mélangent à la même hauteur : le modèle ne sépare pas → features.
Si la colonne 1 est en haut et la 0 en bas : le récit porte, on règle surtout le seuil.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">4. Curseur Train / CV / Test → lequel pour les features ?</p>
<p style="margin:0 0 12px;">
Les trois onglets de données ne servent pas au même moment du travail.
</p>
<table style="width:100%;border-collapse:collapse;margin:0 0 14px;font-size:16px;">
  <tr style="background:#f4eee6;">
    <th style="text-align:left;padding:8px 10px;">Curseur</th>
    <th style="text-align:left;padding:8px 10px;">Pour décider d’une feature ?</th>
    <th style="text-align:left;padding:8px 10px;">Pour le reste</th>
  </tr>
  <tr>
    <td style="padding:8px 10px;"><strong>Train</strong></td>
    <td style="padding:8px 10px;">Oui, en premier : matrices, pairplot, permutation, SHAP global. Plus de lignes → les jumelles et le bloc carrière se voient mieux. C’est là qu’on invente ou qu’on retire une colonne.</td>
    <td style="padding:8px 10px;">Trop beau pour juger Recall / seuil : il a déjà vu ces gens.</td>
  </tr>
  <tr>
    <td style="padding:8px 10px;"><strong>CV</strong></td>
    <td style="padding:8px 10px;">Utile si une feature « marche » seulement sur un pli. Instable d’un pli à l’autre = signal fragile, on ne la crée pas.</td>
    <td style="padding:8px 10px;">SHAP CV = encore le modèle final (pas un modèle par pli).</td>
  </tr>
  <tr>
    <td style="padding:8px 10px;"><strong>Test</strong></td>
    <td style="padding:8px 10px;">Contrôle, pas création. À la fin seulement : après avoir choisi les features sur le train.
On y lit le seuil, la matrice, Recall / Precision.
On n’y invente pas de colonne.</td>
    <td style="padding:8px 10px;">Seuil, PR, matrices, histos : verdict métier.</td>
  </tr>
</table>
<p style="margin:0 0 16px;">
Donc : <strong>on fabrique et on trie les features sur le train</strong> (et la CV si doute),
on <strong>valide le paquet et le seuil sur le test</strong>.
Pas l’inverse → sinon on sculpte le modèle pour 20 % de hasard.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">5. Onglet corrélations (features ON/OFF)</p>
<p style="margin:0 0 16px;">
Matrice = Visualiser les redondances , mais aussi les features qui peuvent être synthétisée, exemple créer la feature jeune_faible_anciennete a partir de age et de ancienneté ou revenu_par_niveau a partir de salaire et de niveau.
<br>
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">6. SHAP global et local</p>
<p style="margin:0 0 16px;">
Beeswarm (global) : droite Part, gauche Reste, Ça répond a la question : en général, le modèle s’accroche à quoi ?
Waterfall (local) : une personne, pour RH.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">7. Revenir aux features</p>
<p style="margin:0 0 16px;">
Pastille rouge, histos train ≠ test, paire encore trop rouge, SHAP accroché à une case minuscule
→ panneau, plutôt après lecture <em>train</em> : décocher une jumelle, garder sat min,
relancer split → préprocesseur → modèle.
Puis <em>test</em> : pastille + PR + seuil. On s’arrête quand c’est stable, calibré, lisible.
</p>

<p style="margin:0;padding-top:16px;border-top:1px solid #e6dfd4;color:black;font-size:17px;">
Train = atelier features. CV = le signal est-il fragile. Test = jugement.
</p>
</div>

# 12. Recherche d'hyper parametres
<div id="analyse-shap"></div>

### → **1**-Actualiser la liste des modèles entraînés  → **2**-Choisir les modèles à analyser → **3**-Récupérer le seuil de l'entrainement ou ajuster  → **4** -Cliquer sur 🚀 **Optimisation automatique** → **5**-cliquer sur **Analyse des features 🔎**

In [12]:
import optimisations__analyses.hyperparameters_interface as hyperparameters_interface
from models__analyses.model import obtenir_modeles

dict_modeles_base = obtenir_modeles()

categories_modeles = {
    "Référence (Baseline)": ["Dummy_Stratified"],
    "Linéaires & Régularisés": [
        "LogisticRegression_None", "LogisticRegression_L1",
        "LogisticRegression_L2", "Ridge", "Lasso_LogReg", "ElasticNet_LogReg",
    ],
    "Arbres & Boosting": [
        "DecisionTree", "RandomForest", "GradientBoosting", "AdaBoost", "XGBoost",
    ],
    "Autres (SVM, KNN)": [
        "SVC_Prob", "SVC_RBF", "SVC_Linear", "SVC_Poly", "SVR_Linear", "KNN",
    ],
}

display(hyperparameters_interface.interface_tuning(dict_modeles_base, categories_modeles))

# 13. Analyse Shap

In [13]:
sys.path.append(os.path.abspath('src/Modules'))

display(shap_analyse.interface_analyse_shap())

<div style="font-size:19px;line-height:1.65;color:#1c2333;background:#fffdf9;border:1px solid #e6dfd4;border-radius:16px;padding:32px 36px;font-family:Georgia,Palatino,serif;">

<p style="margin:0 0 8px;font-family:'Segoe UI',sans-serif;font-size:13px;letter-spacing:.14em;text-transform:uppercase;color:#c45c2a;font-weight:700;">
Attrition → ce que le modèle dit vraiment
</p>
<p style="margin:0 0 22px;font-family:'Segoe UI',sans-serif;font-size:22px;font-weight:700;color:#1c2333;">
Quatre cases, deux curseurs, une courbe
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">1. Dans notre cas : qui est « positif » ?</p>
<p style="margin:0 0 16px;">
La classe positive, c’est <strong>le départ</strong> (<code>a_quitte_l_entreprise = 1</code>).
Une <em>alerte</em> = le modèle prédit " va partir ".
Une <em>non-alerte</em> = " reste ".
</p>

<table style="width:100%;border-collapse:collapse;margin:0 0 20px;font-size:16px;">
  <tr style="background:#f4eee6;">
    <th style="text-align:left;padding:10px 12px;border-bottom:1px solid #e6dfd4;">Case</th>
    <th style="text-align:left;padding:10px 12px;border-bottom:1px solid #e6dfd4;">Réalité</th>
    <th style="text-align:left;padding:10px 12px;border-bottom:1px solid #e6dfd4;">Alerte ?</th>
    <th style="text-align:left;padding:10px 12px;border-bottom:1px solid #e6dfd4;">Chez nous</th>
  </tr>
  <tr>
    <td style="padding:10px 12px;border-bottom:1px solid #f0ebe3;"><strong>VP — vrai positif</strong></td>
    <td style="padding:10px 12px;border-bottom:1px solid #f0ebe3;">est parti</td>
    <td style="padding:10px 12px;border-bottom:1px solid #f0ebe3;">oui</td>
    <td style="padding:10px 12px;border-bottom:1px solid #f0ebe3;">départ attrapé → entretien RH utile</td>
  </tr>
  <tr>
    <td style="padding:10px 12px;border-bottom:1px solid #f0ebe3;"><strong>FP → faux positif</strong></td>
    <td style="padding:10px 12px;border-bottom:1px solid #f0ebe3;">est resté</td>
    <td style="padding:10px 12px;border-bottom:1px solid #f0ebe3;">oui</td>
    <td style="padding:10px 12px;border-bottom:1px solid #f0ebe3;">fausse alerte → temps RH perdu</td>
  </tr>
  <tr>
    <td style="padding:10px 12px;border-bottom:1px solid #f0ebe3;"><strong>FN → faux négatif</strong></td>
    <td style="padding:10px 12px;border-bottom:1px solid #f0ebe3;">est parti</td>
    <td style="padding:10px 12px;border-bottom:1px solid #f0ebe3;">non</td>
    <td style="padding:10px 12px;border-bottom:1px solid #f0ebe3;">départ raté → on n’a rien vu</td>
  </tr>
  <tr>
    <td style="padding:10px 12px;"><strong>VN → vrai négatif</strong></td>
    <td style="padding:10px 12px;">est resté</td>
    <td style="padding:10px 12px;">non</td>
    <td style="padding:10px 12px;">silence juste → rien à faire</td>
  </tr>
</table>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">2. La matrice de confusion</p>
<p style="margin:0 0 16px;">
C’est le <strong>compte des 4 cases</strong> après un seuil choisi.
Elle ne juge pas le modèle « en général » : elle juge <em>cette</em> règle
« au-dessus du seuil → alerte ».
Changer le seuil, c’est faire glisser des gens d’une case à l’autre :
plus d’alertes → plus de VP <em>et</em> plus de FP ; moins d’alertes → plus de FN.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">3. Les indicateurs → quoi tourner pour quoi</p>
<p style="margin:0 0 12px;">
<strong>Recall (rappel)</strong> = VP / (VP + FN) → part des <em>vrais départs</em> qu’on attrape.<br>
On le monte si on veut <strong>rater le moins de départs possible</strong> (seuil plus bas).
Contrepartie : plus de fausses alertes.
</p>
<p style="margin:0 0 12px;">
<strong>Precision</strong> = VP / (VP + FP) → part des <em>alertes</em> qui sont de vrais départs.<br>
On le monte si on veut <strong>ne pas noyer les managers</strong> (seuil plus haut).
Contrepartie : on laisse partir des gens sans alerte.
</p>
<p style="margin:0 0 12px;">
<strong>F1</strong> = compromis Recall ↔ Precision. Utile quand on n’a pas encore tranché RH.<br>
<strong>Accuracy</strong> : à éviter ici. Avec ~16 % de départs, « tout le monde reste » score déjà ~84 %.
</p>
<p style="margin:0 0 16px;">
<strong>ROC-AUC</strong> : le modèle sépare-t-il les profils, <em>tous seuils confondus</em>.<br>
<strong>PR-AUC</strong> : même idée, mais en se concentrant sur la classe rare (les départs). Plus parlant pour l’attrition.
</p>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">4. Combinaisons métier</p>
<ul style="margin:0 0 16px;padding-left:20px;">
  <li style="margin-bottom:8px;"><strong>Filet large</strong> — Recall haut, Precision basse.<br>
  Seuil bas (~0,30–0,40). Beaucoup de VP, beaucoup de FP. Campagne large, 1-to-1 impossible.</li>
  <li style="margin-bottom:8px;"><strong>Filet serré</strong> — Precision haute, Recall plus bas.<br>
  Seuil haut (~0,60–0,75). Peu d’alertes, souvent justes. Entretiens ciblés, des départs invisibles.</li>
  <li style="margin-bottom:8px;"><strong>Compromis</strong> — F1 max, ou Recall ~0,60–0,65 et Precision ~0,40.<br>
  Seuil vers 0,45–0,55 (à lire sur <em>la</em> courbe). Assez de départs vus, assez peu d’alertes pour agir.</li>
</ul>

<p style="margin:0 0 10px;font-family:'Segoe UI',sans-serif;font-size:16px;font-weight:700;color:#c45c2a;">5. La courbe précision–rappel (pour le seuil)</p>
<p style="margin:0 0 16px;">
Axe X = Recall. Axe Y = Precision. Chaque point = <strong>un seuil</strong>.<br>
On avance vers la droite : on attrape plus de départs, la précision tombe.<br>
La ligne pointillée = taux réel de départs (~0,16) : en dessous, le modèle ne vaut pas mieux qu’alerter au hasard.<br>
On ne cherche <em>pas</em> le croisement des trois courbes pour le plaisir : on choisit le point
qui correspond au budget RH (combien d’entretiens on peut tenir).
Le trait rouge de l’onglet « seuil » marque souvent le F1 max — un bon départ, pas un dogme.
</p>

<p style="margin:0;padding-top:16px;border-top:1px solid #e6dfd4;color:black;font-size:17px;">
Ordre pratique : regarder la <strong>matrice</strong> (combien d’alertes, combien de ratés) →
lire la <strong>courbe PR</strong> (où ça casse) → fixer le <strong>seuil</strong> →
revérifier Precision / Recall / F1 sur le test. Le modèle calibre les scores ;
le seuil décide qui reçoit l’alerte.
</p>
</div>